# Session 20b — Multi-organism ESM-2 embeddings

Reads the unified dataset produced by `curate_multiorg_session20.ipynb` (`memory_bank/data/multiorg_essentiality/labels.csv` + `sequences.fasta`), embeds every protein with **ESM-2 650M** at **batch size 16** (Session 19 confirmed both choices), and writes:

- `cell_sim/features/cache/esm2_650M_multiorg.parquet` — index `'{organism}|{locus_tag}'`, columns `esm2_650M_dim_0` … `esm2_650M_dim_1279`

**Pre-requisite:** `curate_multiorg_session20.ipynb` must have run + pushed first. Cell 5 of this notebook hard-fails with a clear error if `labels.csv` or `sequences.fasta` is absent, rather than silently embedding only Syn3A.

**Defenses against Session-16-style failures:**
- **Pre-flight on 8 sequences** validates the model load, fp16 LM path, mean-pool, and finiteness of the resulting embeddings BEFORE the 10k-protein loop starts.
- **Checkpoint every 256 proteins** to `/content/multiorg_embed_checkpoint.pkl`; re-running the cell skips already-embedded composite keys.
- **Mean L2-norm check** at the end — every row must have a finite, non-zero norm (NaN-poisoned rows would hide a model that produced gibberish).

**Wall estimate** (Session 19 measured 119 seq/s on RTX PRO 6000 Blackwell at fp16): ~10,135 / 119 = ~85 s raw inference, plus tokenization + padding + model load = **5-15 min total on A100/L4-class GPUs**, **30-60 min on T4**.


In [ ]:
# Cell 1 — install GPU-side deps + verify CUDA.
!pip install -q \
    "torch>=2.2" \
    "transformers>=4.40" \
    "biopython>=1.83" \
    "pyarrow>=14" \
    "pandas>=2" \
    "numpy>=1.24"

import torch
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {props.total_memory / 1024**3:.1f} GiB")
else:
    raise RuntimeError(
        "This notebook requires a GPU. Runtime menu "
        "-> Change runtime type -> A100 / L4 / T4."
    )

In [ ]:
# Cell 2 — clone (or refresh) the project repo to branch tip.
import os, subprocess

BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"
REPO_URL = "https://github.com/Nikku03/cell.git"
REPO_DIR = "/content/cell"

def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip())
    if r.returncode != 0:
        raise RuntimeError(f"{cmd!r} exit {r.returncode}")
    return r

if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    _run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    _run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    _run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)

%cd /content/cell
import sys
sys.path.insert(0, "/content/cell")
_run(["git", "log", "--oneline", "-1"], cwd=REPO_DIR)

In [ ]:
# Cell 3 — prompt for GitHub PAT (masked).
import os, getpass
if os.environ.get("GITHUB_PAT", "").strip():
    print(f"GITHUB_PAT already set ({len(os.environ['GITHUB_PAT'])} chars).")
else:
    pat = getpass.getpass(
        "Paste your GitHub fine-grained PAT (input hidden): "
    ).strip()
    if not pat:
        raise ValueError("empty PAT — push later will fail")
    os.environ["GITHUB_PAT"] = pat
    print(f"GITHUB_PAT set ({len(pat)} chars).")

In [ ]:
# Cell 4 — load labels.csv + sequences.fasta from the curation step.
#
# Hard-fails if either file is missing. Composite key here MUST match
# the format the trainer expects: '{organism}|{locus_tag}'.
from pathlib import Path
import pandas as pd
from Bio import SeqIO

LABELS = Path("memory_bank/data/multiorg_essentiality/labels.csv")
FASTA = Path("memory_bank/data/multiorg_essentiality/sequences.fasta")
if not LABELS.exists() or not FASTA.exists():
    raise FileNotFoundError(
        f"missing curated dataset files. Run "
        f"notebooks/curate_multiorg_session20.ipynb on Colab + push "
        f"first, then re-run this cell.\n"
        f"  labels.csv exists: {LABELS.exists()}\n"
        f"  sequences.fasta exists: {FASTA.exists()}"
    )

labels = pd.read_csv(LABELS)
labels["_key"] = labels["organism"] + "|" + labels["locus_tag"]
print(f"labels.csv: {len(labels)} rows  cols: {list(labels.columns)}")
print(labels.groupby("organism").size().to_string())

seqs: dict[str, str] = {}
for rec in SeqIO.parse(FASTA, "fasta"):
    # Header: 'organism|locus_tag|gene_name'
    parts = rec.id.split("|")
    if len(parts) < 2:
        continue
    key = f"{parts[0]}|{parts[1]}"
    seqs[key] = str(rec.seq).strip().upper().rstrip("*")
print(f"\nsequences.fasta: {len(seqs)} records")

# Join.
labels["sequence"] = labels["_key"].map(seqs)
missing = labels[labels["sequence"].isna()]
if len(missing):
    print(f"\n[warn] {len(missing)} labels rows have no fasta record; "
          f"first 5: {missing['_key'].head().tolist()}")
labels = labels.dropna(subset=["sequence"]).reset_index(drop=True)
labels = labels[labels["sequence"].str.len() > 0].reset_index(drop=True)
print(f"\nrows ready to embed: {len(labels)}  "
      f"len_mean={int(labels['sequence'].str.len().mean())}aa  "
      f"len_max={int(labels['sequence'].str.len().max())}aa")

In [ ]:
# Cell 5 — load ESM-2 650M (fp16 LM weights on GPU).
#
# Pure embedding — no structure prediction, so the Session-16 fp16
# pLDDT-NaN bug doesn't apply here. We can safely run the LM in fp16.
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_ID = "facebook/esm2_t33_650M_UR50D"
EMBED_DIM = 1280
BATCH_SIZE = 16        # Session 19 measured peak
MAX_SEQ_LEN = 1022     # ESM-2's hard cap minus BOS/EOS

print(f"loading {MODEL_ID} ...")
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to("cuda").eval().half()
torch.cuda.synchronize()
print("model loaded; ready for inference")

In [ ]:
# Cell 6 — pre-flight on the first 8 sequences.
#
# Validates that the model + tokenizer + mean-pool produces finite,
# non-zero embeddings BEFORE we commit to a 10k-protein loop. Catches
# fp16 NaN, padding-mask bugs, and tokenizer surprises (extra tokens,
# uppercase requirement) cheaply.
import numpy as np

PRE_N = 8
pre_seqs = labels["sequence"].head(PRE_N).tolist()
print(f"[pre-flight] embedding {PRE_N} sequences...")
with torch.no_grad():
    enc = tok(
        pre_seqs, padding=True, truncation=True,
        max_length=MAX_SEQ_LEN, return_tensors="pt",
    ).to("cuda")
    out = model(**enc)
    mask = enc["attention_mask"].unsqueeze(-1).float()
    summed = (out.last_hidden_state * mask).sum(dim=1)
    pooled = (summed / mask.sum(dim=1)).float().cpu().numpy()

norms = np.linalg.norm(pooled, axis=1)
print(f"  shape: {pooled.shape}")
print(f"  L2 norms: min={norms.min():.2f} max={norms.max():.2f} "
      f"mean={norms.mean():.2f}")
assert pooled.shape == (PRE_N, EMBED_DIM), pooled.shape
assert np.all(np.isfinite(pooled)), "NaN / Inf in pre-flight pool"
assert norms.min() > 0.1, f"zero-norm row in pre-flight: {norms}"
print("[pre-flight] OK — proceeding with full embed loop")

In [ ]:
# Cell 7 — checkpointed embed loop.
#
# Iterates labels in order, embeds in batches of BATCH_SIZE, pickles
# a (key -> embedding) dict every CHECKPOINT_EVERY composite keys.
# Re-running the cell after a crash skips already-embedded keys.
import pickle
import time
from pathlib import Path

import numpy as np
import torch

CHECKPOINT_PATH = Path("/content/multiorg_embed_checkpoint.pkl")
CHECKPOINT_EVERY = 256

if CHECKPOINT_PATH.exists():
    with CHECKPOINT_PATH.open("rb") as fh:
        ckpt = pickle.load(fh)
    print(f"[resume] checkpoint with {len(ckpt['rows'])} embeddings already done")
else:
    ckpt = {"rows": {}, "started_at": time.time()}

todo = labels[~labels["_key"].isin(ckpt["rows"])].reset_index(drop=True)
print(f"to embed: {len(todo)}/{len(labels)}")

t0 = time.perf_counter()
with torch.no_grad():
    for i in range(0, len(todo), BATCH_SIZE):
        chunk = todo.iloc[i:i + BATCH_SIZE]
        enc = tok(
            chunk["sequence"].tolist(),
            padding=True, truncation=True,
            max_length=MAX_SEQ_LEN, return_tensors="pt",
        ).to("cuda")
        out = model(**enc)
        mask = enc["attention_mask"].unsqueeze(-1).float()
        summed = (out.last_hidden_state * mask).sum(dim=1)
        pooled = (summed / mask.sum(dim=1)).float().cpu().numpy()
        for k, vec in zip(chunk["_key"], pooled):
            ckpt["rows"][k] = vec.astype(np.float32)

        done = len(ckpt["rows"])
        if done % CHECKPOINT_EVERY < BATCH_SIZE or (i + BATCH_SIZE) >= len(todo):
            with CHECKPOINT_PATH.open("wb") as fh:
                pickle.dump(ckpt, fh)
            elapsed = time.perf_counter() - t0
            done_now = i + len(chunk)
            rate = done_now / max(1e-6, elapsed)
            eta_s = (len(todo) - done_now) / max(1e-6, rate)
            print(f"  [ckpt] {done}/{len(labels)} embedded  "
                  f"this-cell elapsed={elapsed:.0f}s  "
                  f"eta={eta_s:.0f}s  rate={rate:.1f} seq/s")

embed_wall = time.perf_counter() - t0
print(f"\nembed loop done in {embed_wall:.1f}s; "
      f"total embeddings on hand: {len(ckpt['rows'])}")

In [ ]:
# Cell 8 — assemble + write parquet.
#
# Build a DataFrame with the labels' row order, then sanity-check
# every row has a finite, non-zero L2 norm before saving.
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

EMBED_PARQUET = Path("cell_sim/features/cache/esm2_650M_multiorg.parquet")
EMBED_PARQUET.parent.mkdir(parents=True, exist_ok=True)

missing_keys = [k for k in labels["_key"] if k not in ckpt["rows"]]
if missing_keys:
    raise RuntimeError(
        f"{len(missing_keys)} keys missing from checkpoint; "
        f"first 5: {missing_keys[:5]}. Re-run Cell 7."
    )

feature_cols = [f"esm2_650M_dim_{i}" for i in range(EMBED_DIM)]
rows_arr = np.stack([ckpt["rows"][k] for k in labels["_key"]], axis=0)
norms = np.linalg.norm(rows_arr, axis=1)
n_finite = int(np.isfinite(rows_arr).all(axis=1).sum())
n_nonzero = int((norms > 0.1).sum())
print(f"finite rows: {n_finite}/{len(labels)}")
print(f"non-zero-norm rows: {n_nonzero}/{len(labels)}")
print(f"L2 norms: min={norms.min():.2f} max={norms.max():.2f} "
      f"mean={norms.mean():.2f}")
if n_finite != len(labels) or n_nonzero != len(labels):
    raise RuntimeError(
        "some rows are NaN/zero — model produced bad embeddings"
    )

feats = pd.DataFrame(rows_arr, columns=feature_cols)
feats.index = pd.Index(labels["_key"], name="_key")
feats.to_parquet(EMBED_PARQUET)
sha = hashlib.sha256(EMBED_PARQUET.read_bytes()).hexdigest()
size_mib = EMBED_PARQUET.stat().st_size / 1024**2
print(f"\nwrote {EMBED_PARQUET}")
print(f"  rows={len(feats)}  cols={len(feature_cols)}")
print(f"  sha256={sha}")
print(f"  size={size_mib:.1f} MiB")

# Spot-check: print a few composite keys + their norms grouped
# by organism so the reviewer can eyeball coverage.
import re
by_org = {}
for k, n in zip(feats.index, norms):
    o = k.split("|", 1)[0]
    by_org.setdefault(o, []).append(n)
print("\nper-organism norm summary:")
for o in sorted(by_org):
    arr = np.array(by_org[o])
    print(f"  {o:6s}  n={len(arr):<5d}  mean_norm={arr.mean():.2f}  "
          f"std_norm={arr.std():.2f}")

In [ ]:
# Cell 9 — auto-commit + push.
import os, subprocess

pat = os.environ.get("GITHUB_PAT", "").strip()
if not pat:
    raise SystemExit("GITHUB_PAT not set; re-run Cell 3.")

BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"
MSG = (
    "Session 20b: ESM-2 650M multi-organism embeddings "
    "(esm2_650M_multiorg.parquet, ~10k rows)"
)

def run(cmd, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip())
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd!r} exit {r.returncode}")
    return r

run(["git", "config", "user.email", "cell-sim-bot@noreply.local"])
run(["git", "config", "user.name", "cell-sim-bot"])
run(["git", "add", "-f",
     "cell_sim/features/cache/esm2_650M_multiorg.parquet"])

status = run(["git", "status", "--porcelain"], check=False)
if not status.stdout.strip():
    print("nothing changed")
else:
    run(["git", "commit", "-m", MSG])
    remote = f"https://{pat}@github.com/Nikku03/cell.git"
    run(["git", "push", remote, BRANCH])
    print("\npush complete.")

# Clean up the checkpoint once the parquet lands so a re-run
# with a different curated dataset rebuilds from scratch.
from pathlib import Path
ckpt_path = Path("/content/multiorg_embed_checkpoint.pkl")
if ckpt_path.exists():
    ckpt_path.unlink()
    print(f"removed {ckpt_path}")

run(["git", "log", "--oneline", "-3"])